In [1]:
from __future__ import annotations
from collections.abc import Callable

In [ ]:
class Value:
    """
    Building block to store indiviual data, operation, track gredients, enables backprop
    """
    def __init__(self, data: float, name: str = "", grad: float = 0, child: tuple[Value, Value] | None = None ) -> None:
        """
        Node initialization, record
            - current node's data
            - current node's gradient, initialized as 0
            - initialize place holder for parent node, initialize as None
            - add current node name for tracing, not related to functionality of the object, initialized as empty string

        Note:
            - when we have `a * b = c`, `c` is the parent of `a` and `b`, this is how this module makes reference 
            - make `_back()` defined instead of `None`, so later doesn't have to check whether the closure exist before calling it

        Important:
            - should track both children instead of one parent
        """
        self.data = data
        self.name = name
        self.grad = grad
        self.child = child
        def _back() -> None:
            pass 
        self._back: Callable[[], None] = _back

    def __repr__(self) -> str:
        child = (self.child[0].name, self.child[1].name) if self.child else "Empty"
        return (
            f"Value Class for '{self.name}': \n"
            f"  data: {self.data}\n"
            f"  gradient: {self.grad}\n"
            f"  children: {child}\n"
        )

    def __mul__(self, y: Value) -> Value:
        """
        Applying multiplication to another number, returns another Value object
            - current node's parent is the resulting 'Value' object
            - current node's grad is (the number being mulipled with) x (parent's gradient)

        Important:
            - should implement a 'rule' instead of computing gradient on the spot
            - modify the method with regard to children instead of parent
        """
        y_data = y.data
        result = Value(data = self.data * y_data, child=(self, y))
        # result.child = (self, y)
        # define closure to remember how to backpropogate when called
        def _back():
            parent_grad = result.grad
            self.grad = y_data * parent_grad
            y.grad = self.data * parent_grad
        result._back = _back
        return result

In [12]:
a = Value(3, "a")
b = Value(4, "b")
print(a)

Value Class for 'a': 
  data: 3
  gradient: 0
  children: Empty



In [14]:
c = a * b
c.name = "c"
c.grad = 1
print(c)

Value Class for 'c': 
  data: 12
  gradient: 1
  children: ('a', 'b')



In [15]:
c._back()

In [17]:
print(a)

Value Class for 'a': 
  data: 3
  gradient: 4
  children: Empty



In [18]:
print(b)

Value Class for 'b': 
  data: 4
  gradient: 3
  children: Empty

